# RQ1 ? Evaluation Kaggle sulle 104 foto comuni

Notebook preparato il 12 novembre 2026. **Non ? stato eseguito su Kaggle.**

Valuta **3.744 immagini gi? generate**: 104 foto ? 3 stili ? 4 condizioni ? 3 seed.
Non genera immagini, non carica Stable Diffusion/ControlNet/LoRA e non usa il computer locale.
Le metriche sono CLIP-I, CLIP-T, DINO-I e mIoU semantica. Il codice GPU usa soltanto i valutatori.

## Prima di iniziare

1. Importa questo `.ipynb` in un notebook Kaggle e seleziona una GPU.
2. Allega un dataset con **l'intera struttura di `result_rq1/`**, mantenendo i cinque shard.
3. Allega anche `rq1_evaluation_code.zip` fornito insieme al notebook, per usare il codice aggiornato.
4. Abilita Internet per installare le dipendenze di evaluation e scaricare i tre checkpoint pubblici.
5. Imposta i due percorsi nella prima cella. Esegui le celle in ordine.

Lascia **`APPROVA_INFERENZA = False`** fino al successivo OK concordato: la cella GPU si fermer?.
Le celle precedenti eseguono preparazione, controlli di file e download delle revisioni/configurazioni;
la copia delle immagini usa disco e CPU, non inferenza. Non usare `Run All` prima di aver letto le celle.

Gli originali restano in sola lettura. Tutti i nuovi file sono sotto `/kaggle/working/`.


In [ ]:
from pathlib import Path
import sys, json, hashlib, subprocess, shutil, csv, os, platform
from datetime import datetime, timezone

# Sostituire con i percorsi reali mostrati nei dataset allegati.
BUNDLE_PATH = Path('/kaggle/input/NOME_DATASET_CODICE/rq1_evaluation_code.zip')
SOURCE_ROOT = Path('/kaggle/input/NOME_DATASET_RISULTATI/result_rq1')
APPROVA_INFERENZA = False  # Cambiare solo dopo il successivo OK per il lancio.

assert sys.version_info >= (3, 11), 'Richiesto Python >= 3.11'
assert Path('/kaggle/working').is_dir(), 'Questo notebook va eseguito su Kaggle.'
assert BUNDLE_PATH.is_file(), f'Correggere BUNDLE_PATH: {BUNDLE_PATH}'
assert (SOURCE_ROOT / 'baseline_seed_123/run_config.json').is_file(), 'Correggere SOURCE_ROOT.'
SESSION = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
WORK = Path('/kaggle/working') / f'rq1_common104_{SESSION}'
WORK.mkdir(exist_ok=False)
CODE_ROOT = WORK / 'code'
RESULTS = WORK / 'results'
RESULTS.mkdir()
SUBSET = WORK / 'selected_runs'
SEEDS = (42, 123, 456)
print('Cartella sessione:', WORK)
print('Inferenza autorizzata:', APPROVA_INFERENZA)


## 1. Estrarre il codice e installare solo le dipendenze di evaluation

Si riusa il PyTorch CUDA fornito dall'ambiente Kaggle; non si installa l'intero
`requirements.txt`, che include anche training e generazione. L'installazione modifica
soltanto l'ambiente di questa sessione Kaggle. Se pip chiede un riavvio del kernel,
riavviare e ripartire dalla configurazione: verr? creata una nuova sessione di output.


In [ ]:
import zipfile
EXPECTED_BUNDLE_SHA256 = 'c8d85bf999e7d0cf889572e0bec3a72830bd18f9a2a15126bcaab25ccfa89c57'
actual_bundle_hash = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()
assert actual_bundle_hash == EXPECTED_BUNDLE_SHA256, 'Bundle diverso da quello verificato per questo notebook.'
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    for name in archive.namelist():
        target = (CODE_ROOT / name).resolve()
        assert target.is_relative_to(CODE_ROOT.resolve()), 'Percorso non valido nello ZIP.'
    archive.extractall(CODE_ROOT)
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'transformers==4.57.6', 'huggingface-hub>=0.34,<1',
                'numpy>=1.26,<3', 'Pillow>=10,<14', 'safetensors>=0.4,<1'], check=True)


## 2. Registrare l'ambiente e congelare i valutatori

Controlla che la GPU Kaggle sia disponibile, senza eseguire inferenze.
Risolviamo le revisioni `main` in commit Hugging Face immutabili **una sola volta**,
poi usiamo lo stesso TOML congelato per tutti i seed. Nessun testo CLIP-T viene modificato.
Il TOML e l'ambiente saranno inclusi nell'archivio finale.


In [ ]:
import torch, tomllib, re
from huggingface_hub import HfApi
from importlib import metadata
assert torch.cuda.is_available(), 'GPU Kaggle non disponibile: selezionarla prima di proseguire.'
selection_path = CODE_ROOT / 'configs/rq1_common104.json'
selection = json.loads(selection_path.read_text(encoding='utf-8'))
assert len(selection['image_ids']) == 104 and selection['expected_outputs'] == 3744
config_text = (CODE_ROOT / 'configs/evaluation.toml').read_text(encoding='utf-8')
protocol = tomllib.loads(config_text)
resolved = {}
for section in ('clip', 'dino', 'semantic'):
    item = protocol[section]
    resolved[section] = HfApi().model_info(item['model_id'], revision=item['revision']).sha
    assert resolved[section] and re.fullmatch(r'[0-9a-f]{40}', resolved[section])
    pattern = rf'(\[{section}\][^\[]*?revision\s*=\s*)"[^"]+"'
    config_text, count = re.subn(pattern, lambda m: m.group(1) + json.dumps(resolved[section]), config_text, count=1)
    assert count == 1
PROTOCOL = RESULTS / 'evaluation_frozen.toml'
PROTOCOL.write_text(config_text, encoding='utf-8')
shutil.copy2(selection_path, RESULTS / 'selection.json')
environment = {
    'python': sys.version, 'platform': platform.platform(),
    'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__,
    'cuda': torch.version.cuda, 'bundle_sha256': actual_bundle_hash,
    'selection_sha256': hashlib.sha256(selection_path.read_bytes()).hexdigest(),
    'protocol_sha256': hashlib.sha256(PROTOCOL.read_bytes()).hexdigest(),
    'resolved_revisions': resolved,
    'packages': {p: metadata.version(p) for p in ('transformers','huggingface-hub','numpy','Pillow')},
    'source_root': str(SOURCE_ROOT), 'session_utc': SESSION,
}
(RESULTS / 'environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
print(json.dumps(environment, indent=2))


## 3. Preparare le tre copie selezionate ? senza inferenza

Verifica configurazioni, metadati, duplicati, presenza dei 3.744 PNG e hash degli input.
Stima lo spazio richiesto, poi copia soltanto input, output e metadati necessari alle metriche.
Non copia le mappe ControlNet. Conserva la provenienza degli shard e gli hash dei PNG copiati.
Sono attesi **104 input e 1.248 output per seed**.

La copia pu? richiedere tempo. Se si interrompe, non riutilizzare una cartella parziale:
ripartire con una nuova sessione. Non cancellare gli originali.


In [ ]:
from src.evaluation.prepare_subset import plan_subset
_, planned_rows = plan_subset(SOURCE_ROOT, selection)
unique_inputs = {(key[0], key[3]): item[1] for key, item in planned_rows.items()}
copy_bytes = sum(item[2].stat().st_size for item in planned_rows.values())
copy_bytes += sum(path.stat().st_size for path in unique_inputs.values())
free_bytes = shutil.disk_usage(WORK).free
print(f'Copia immagini: circa {copy_bytes / 2**30:.2f} GiB; liberi: {free_bytes / 2**30:.2f} GiB')
assert free_bytes > copy_bytes + 3 * 2**30, 'Spazio insufficiente per copie, risultati e cache dei modelli.'

def run_logged(arguments, log_path):
    # Comandi strutturati, nessuna shell; un processo separato libera la VRAM al termine.
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen([sys.executable, *map(str, arguments)], cwd=CODE_ROOT,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', errors='replace', bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        status = process.wait()
    if status:
        raise RuntimeError(f'Exit code {status}; consultare {log_path}')

if SUBSET.exists():
    raise RuntimeError('Copie gi? presenti: non sovrascrivere. Se complete, proseguire con audit; altrimenti nuova sessione.')
run_logged(['-m', 'src.evaluation.prepare_subset', '--source-root', SOURCE_ROOT,
            '--selection', selection_path, '--output-dir', SUBSET], RESULTS / 'prepare.log')
shutil.copy2(SUBSET / 'provenance.json', RESULTS / 'provenance.json')


## 4. Audit dei tre seed ? senza modelli

Ogni audit deve avere **1.248 output pronti, 0 invalidi, 104 foto**, tre stili,
quattro condizioni e il seed atteso. Si controlla inoltre che gli hash dei PNG copiati
corrispondano alla provenienza. Se un controllo fallisce, fermarsi prima delle metriche.


In [ ]:
from src.evaluation.prepare_subset import sha256
provenance = json.loads((RESULTS / 'provenance.json').read_text(encoding='utf-8'))
assert len(provenance) == 3744
for record in provenance:
    output = (SUBSET / f"seed_{record['seed']}" / 'samples' / record['image_id']
              / 'images' / record['style'] / record['condition']
              / f"{Path(record['image_id']).stem}_seed{record['seed']}.png")
    assert sha256(output) == record['copied_output_sha256'], f'Copia modificata: {output}'
for seed in SEEDS:
    audit_dir = RESULTS / f'audit_seed_{seed}'
    if not audit_dir.exists():
        run_logged(['-m', 'src.evaluation.evaluate', '--run-dir', SUBSET / f'seed_{seed}',
                    '--config', PROTOCOL, '--audit-only', '--output-dir', audit_dir],
                   RESULTS / f'audit_seed_{seed}.log')
    report = json.loads((audit_dir / 'audit_summary.json').read_text())
    assert report['expected_outputs'] == report['ready_outputs'] == 1248
    assert report['invalid_outputs'] == 0 and report['n_inputs'] == 104
    assert report['seeds'] == [seed] and report['full_rq1_conditions']
    print(seed, ': audit completo')
AUDIT_OK = True
print('Preparazione completa. Fermarsi qui in attesa dell?OK per l?inferenza.')


## 5. BLOCCO ? approvazione prima della GPU

**Questo ? il punto in cui aspettare il successivo OK.**
La preparazione del notebook non autorizza il lancio dell'evaluation.
Dopo l'OK, impostare `APPROVA_INFERENZA = True` nella cella seguente ed eseguirla.
Non ? necessario rieseguire la prima cella, che creerebbe una nuova sessione.

Il primo seed consente di misurare il tempo effettivo. Non stimiamo qui una durata
senza aver osservato GPU e throughput reali. Le tre evaluation sono sequenziali;
ognuna carica CLIP, SegFormer e DINOv2 e calcola tutte le metriche sui suoi 1.248 output.


In [ ]:
# Cambiare soltanto dopo l'OK esplicito per il lancio su Kaggle.
APPROVA_INFERENZA = False
if not APPROVA_INFERENZA:
    raise RuntimeError('Inferenza non autorizzata: fermarsi qui e attendere l?OK.')
assert AUDIT_OK, 'Prima completare tutti gli audit.'


## 6. Evaluation completa ? GPU Kaggle

Non si usa `--allow-partial`: tutte le condizioni devono essere presenti.
Ogni processo salva progressivamente `per_image_metrics.csv`, log e metadati.
Il programma **non ha resume all'interno di un seed**: se un seed si interrompe,
conservare l'output e rilanciarlo in una nuova directory (vedi `ATTEMPT`).
Per riprendere dagli altri seed, la cella salta soltanto evaluation gi? concluse e verificate.
Non modifica automaticamente il subset se una metrica fallisce.


In [ ]:
assert APPROVA_INFERENZA and AUDIT_OK, 'Manca approvazione o audit.'
ATTEMPT = {42: 1, 123: 1, 456: 1}  # Aumentare solo per un seed interrotto; i vecchi file restano.
METRICS = ('clip_i', 'clip_t', 'semantic_miou', 'dino_i')

def check_evaluation(folder, seed):
    report = json.loads((folder / 'evaluation_summary.json').read_text())
    assert report['evaluated_outputs'] == report['successful_outputs'] == 1248
    assert report['complete_image_style_seed_blocks'] == 312
    assert report['outputs_used_in_summary'] == 1248
    config = json.loads((folder / 'evaluation_config.json').read_text())
    assert config['protocol'] == tomllib.loads(PROTOCOL.read_text())
    assert set(config['metrics']) == set(METRICS)
    with (folder / 'per_image_metrics.csv').open(newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    import math
    expected = {(i,s,c,str(seed)) for i in selection['image_ids']
                for s in selection['styles'] for c in selection['conditions']}
    actual = {(r['image_id'],r['style'],r['condition'],r['seed']) for r in rows}
    assert len(rows) == 1248 and actual == expected
    for row in rows:
        assert row['status'] == 'success'
        assert row['input_sha256'] == selection['input_sha256'][row['image_id']]
        assert all(math.isfinite(float(row[m])) for m in METRICS)
    return folder

completed = {}
for seed in SEEDS:
    folder = RESULTS / f'eval_seed_{seed}_attempt{ATTEMPT[seed]}'
    if folder.exists():
        if not (folder / 'evaluation_summary.json').exists():
            raise RuntimeError(f'Tentativo incompleto: {folder}; incrementare ATTEMPT[{seed}] senza cancellarlo.')
        completed[seed] = check_evaluation(folder, seed)
        print(seed, ': gi? completato e verificato, salto il calcolo')
        continue
    run_logged(['-m', 'src.evaluation.evaluate', '--run-dir', SUBSET / f'seed_{seed}',
                '--config', PROTOCOL, '--device', 'cuda', '--metrics', *METRICS,
                '--output-dir', folder], RESULTS / f'eval_seed_{seed}_attempt{ATTEMPT[seed]}.log')
    completed[seed] = check_evaluation(folder, seed)
    print(seed, ': completato e verificato')
(RESULTS / 'completed_evaluations.json').write_text(
    json.dumps({str(k): str(v.relative_to(RESULTS)) for k,v in completed.items()}, indent=2), encoding='utf-8')


## 7. Esportare i risultati

Produce uno ZIP con CSV individuali, riepiloghi per seed, confronti vs `none`,
audit, log, configurazioni, revisioni congelate, selezione, provenienza e codice usato.
Non include copie delle immagini o pesi dei modelli.

Le tabelle restano **per seed**: l'aggregazione statistica finale fra seed e i confronti
fra ControlNet saranno il prossimo passo concordato. Non mediare direttamente le tre
tabelle di riepilogo per costruire gli intervalli di confidenza.

Scaricare lo ZIP dagli output Kaggle. In caso di interruzione si pu? eseguire questa
cella anche senza tutte le metriche: il nome e il manifest indicheranno `PARTIAL`.
Una sessione interrotta non va presentata come evaluation completa.


In [ ]:
verified = {}
for seed in SEEDS:
    candidates = sorted(RESULTS.glob(f'eval_seed_{seed}_attempt*'))
    for folder in candidates:
        if not (folder / 'evaluation_summary.json').exists():
            continue
        try:
            check_evaluation(folder, seed)
        except Exception as error:
            print('Tentativo non valido:', folder.name, str(error))
        else:
            verified[str(seed)] = folder.name
status = 'COMPLETE' if len(verified) == 3 else 'PARTIAL'
(RESULTS / 'export_status.json').write_text(json.dumps({
    'status': status, 'verified_evaluations': verified,
    'expected_total_outputs': 3744, 'selection_id': selection['selection_id'],
}, indent=2), encoding='utf-8')
shutil.copy2(BUNDLE_PATH, RESULTS / 'rq1_evaluation_code.zip')
archive_path = shutil.make_archive(str(WORK / f'rq1_evaluation_{status}_{SESSION}'), 'zip', RESULTS)
print('Stato:', status)
print('Scaricare:', archive_path)
